In [ ]:
#| eval: false
! [ -e /content ] && pip install -Uqq xcube # upgrade xcube on colab

In [ ]:
from fastai.text.all import *
from xcube.text.all import *
from fastai.metrics import accuracy # there's an 'accuracy' metric in xcube as well (Deb fix name conflict later)  

Make sure we have that "beast":

In [ ]:
#| eval: false
ic(torch.cuda.get_device_name(default_device()));
test_eq(torch.cuda.get_device_name(0), torch.cuda.get_device_name(default_device()))
test_eq(default_device(), torch.device(0))
print(f"GPU memory = {torch.cuda.get_device_properties(default_device()).total_memory/1024**3}GB")

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
source = untar_xxx(XURLs.MIMIC3)
# source_l2r = untar_xxx(XURLs.MIMIC3_L2R)

Setting some environment variables:

In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

Setting defaults for pandas and matplotlib:

In [ ]:
# Set the default figure size
plt.rcParams["figure.figsize"] = (8, 4)

Altering some default jupyter settings:

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "last" # "all"

# Training an XML Text Classifier

## `DataLoaders` for the Language Model

To be able to use Transfer Learning, first we need to fine-tune our Language Model (which we pretrained on Wikipedia) on the corpus of Wiki-500k (the one we downloaded). Here we will build the `DataLoaders` object using fastai's `DataBlock` API:

In [ ]:
data = source/'mimic3-9k.csv'
!head -n 1 {data}

In [ ]:
df = pd.read_csv(data,
                 header=0,
                 names=['subject_id', 'hadm_id', 'text', 'labels', 'length', 'is_valid'],
                 dtype={'subject_id': str, 'hadm_id': str, 'text': str, 'labels': str, 'length': np.int64, 'is_valid': bool})
len(df)

In [ ]:
df[['text', 'labels']] = df[['text', 'labels']].astype(str)

Let's take a look at the data:

In [ ]:
df.head(3)

We will now create the `DataLoaders` using `DataBlock` API:

In [ ]:
dls_lm = DataBlock(
    blocks   = TextBlock.from_df('text', is_lm=True),
    get_x    = ColReader('text'),
    splitter = RandomSplitter(0.1)
).dataloaders(df, bs=384, seq_len=80)

For the backward LM:

In [ ]:
dls_lm_r = DataBlock(
    blocks   = TextBlock.from_df('text', is_lm=True, backwards=True),
    get_x    = ColReader('text'),
    splitter = RandomSplitter(0.1)
).dataloaders(df, bs=384, seq_len=80)

Let's take a look at the batches:

In [ ]:
dls_lm.show_batch(max_n=2)

In [ ]:
dls_lm_r.show_batch(max_n=2)

The length of our vocabulary is:

In [ ]:
len(dls_lm.vocab)

Let's take a look at some words of the vocab:

In [ ]:
print(coll_repr(L(dls_lm.vocab), 30))

Creating the `DataLaoders` takes some time, so smash that *save* button (also a good idea to save the `dls_lm.vocab` for later use) if you are working on your own dataset. In this case though `untar_xxx` has got it for you:

In [ ]:
print("\n".join(L(source.glob("**/*dls*lm*.pkl")).map(str))) # the ones with _r are for the reverse language model

To load back the `dls_lm` later on:

In [ ]:
dls_lm = torch.load(source/'mimic3-9k_dls_lm.pkl')

In [ ]:
dls_lm_r = torch.load(source/'mimic3-9k_dls_lm_r.pkl')

## `Learner` for the Language Model Fine-Tuning:

In [ ]:
learn = language_model_learner(
    dls_lm, AWD_LSTM, drop_mult=0.3,
    metrics=[accuracy, Perplexity()]).to_fp16()

And, one more for the reverse:

In [ ]:
learn_r = language_model_learner(
    dls_lm_r, AWD_LSTM, drop_mult=0.3, backwards=True,
    metrics=[accuracy, Perplexity()]).to_fp16()

Training a language model on the full datset takes a lot of time. So you can train one on a tiny dataset for illustration. Or you can skip the training and just load up the one that's pretrained and downloaded by `untar_xxx` and just do the validation.

Let's compute the learning rate using the `lr_find`:

In [ ]:
lr_min, lr_steep, lr_valley, lr_slide = learn.lr_find(suggest_funcs=(minimum, steep, valley, slide))
lr_min, lr_steep, lr_valley, lr_slide

In [ ]:
learn.fit_one_cycle(1, lr_min)

It takes quite a while to train each epoch, so we'll be saving the intermediate model results during the training process:

Since we have completed the initial training, we will now continue fine-tuning the model after unfreezing:

In [ ]:
learn.unfreeze()

and run `lr_find` again, because we now have more layers to train, and the last layers weight have already been trained for one epoch:

In [ ]:
lr_min, lr_steep, lr_valley, lr_slide = learn.lr_find(suggest_funcs=(minimum, steep, valley, slide))
lr_min, lr_steep, lr_valley, lr_slide

Let's now traing with a suitable learning rate:

In [ ]:
learn.fit(15, lr=1e-5, cbs=SaveModelCallback(fname='lm_now'))

Reverse

In [ ]:
learn_r.unfreeze()

In [ ]:
learn_r.fit(20, lr=1e-5, cbs=SaveModelCallback(fname='lm_r_now'))

Note: Make sure if you have trained the most recent language model `Learner` for more epochs (then you need to save that version)

Here you can load the pretrained language model which `untar_xxx` downloaded:

In [ ]:
learn = learn.load(source/'mimic3-9k_lm')

In [ ]:
learn_r = learn_r.load(source/'mimic3-9k_lm_r')

Let's validate the `Learner` to make sure we loaded the correct version:

In [ ]:
learn.validate()

and the reverse...

In [ ]:
learn_r.validate()

### Saving the encoder of the Language Model

**Crucial:** Once we have trained our LM we will save all of our model except the final layer that converts activation to probabilities of picking each token in our vocabulary. The model not including the final layer has a sexy name - *encoder*. We will save it using `save_encoder` method of the `Learner`:

In [ ]:
# learn.save_encoder('lm_finetuned')
# learn_r.save_encoder('lm_finetuned_r')

### Saving the decoder of the Language Model

In [ ]:
learn.save_decoder('mimic3-9k_lm_decoder')
learn_r.save_decoder('mimic3-9k_lm_decoder_r')

This completes the second stage of the text classification process - fine-tuning the Language Model pretrained on Wikipedia corpus. We will now use it to fine-tune a text multi-label text classifier.

## `DataLoaders` for the Multi-Label Classifier (using fastai's Mid-Level Data API)

Fastai's midlevel data api is the swiss army knife of data preprocessing. Detailed tutorials can be found here [intermediate](https://docs.fast.ai/tutorial.pets.html), [advanced](https://docs.fast.ai/tutorial.siamese.html). We will use it here to create our dataloaders for the classifier. 

### Loading Raw Data

In [ ]:
data = source/'mimic3-9k_full.csv'
!head -n 1 {data}

In [ ]:
# !shuf -n 200000 {data} > {data_sample} 

In [ ]:
df = pd.read_csv(data,
                 header=0,
                 names=['subject_id', 'hadm_id', 'text', 'labels', 'length', 'is_valid', 'split'],
                 dtype={'subject_id': str, 'hadm_id': str, 'text': str, 'labels': str, 'length': np.int64, 'is_valid': bool, 'split': str})

In [ ]:
df[['text', 'labels']] = df[['text', 'labels']].astype(str)

In [ ]:
df.head(3)

Sample a small fraction of the dataset to ensure quick iteration (Skip this if you want to do this on the full dataset)

In [ ]:
# df = df.sample(frac=0.3, random_state=89, ignore_index=True)
# df = df.sample(frac=0.025, random_state=89, ignore_index=True)
# df = df.sample(frac=0.005, random_state=89, ignore_index=True)
len(df)

Sample data for low shot experimentation for paper:

In [ ]:
df['is_valid'].value_counts()

In [ ]:
def inst_per_label(df):
    lbl_freqs = Counter()
    for labels in df.labels: lbl_freqs.update(labels.split(';'))
    return array(list(lbl_freqs.values())).mean()

In [ ]:
trn_splts = [25, 50, 100, 200, 400, 800, 1600, 3200, 4500, 6400, 9000, 12800, 18000, 25600, 36000, 49354]
files = [source/('mimic3-9k_' + str(o) + '.csv') for o in trn_splts]
files
for splt,file in zip(trn_splts, files):
    df_sample = df[~df['is_valid']].sample(n=splt)
    print(f"{file = }, instances per label = {inst_per_label(df_sample)}")
    df_sample = pd.concat((df_sample, df[df['is_valid']]))
    df_sample.to_csv(file, index=False)

Let's now gather the labels from the 'labels' columns of the df:

In [ ]:
lbl_freqs = Counter()
for labels in df.labels: lbl_freqs.update(labels.split(';'))

The total number of labels are:

In [ ]:
len(lbl_freqs)

Let's take a look at the most common labels:

In [ ]:
pd.DataFrame(lbl_freqs.most_common(20), columns=['label', 'frequency'])

Let's make a list of all labels (We will use it later while creating the `DataLoader`)

In [ ]:
lbls = list(lbl_freqs.keys())

### Dataset Statistics (Optional) 
> Let's try to understand what captures the hardness of an xml dataset

#### Check #1: Number of instances (train/valid split)

In [ ]:
train, valid = df.index[~df['is_valid']], df.index[df['is_valid']]
len(train), len(valid)

#### Check #2: Avg number of instances per label

In [ ]:
array(list(lbl_freqs.values())).mean()

#### Check #3: Plotting the label distribution

In [ ]:
lbl_count = []
for lbls in df.labels: lbl_count.append(len(lbls.split(';')))

In [ ]:
df_copy = df.copy()
df_copy['label_count'] = lbl_count

In [ ]:
df_copy.head(2)

The average number of labels per instance is:

In [ ]:
df_copy.label_count.mean()

In [ ]:
import seaborn as sns
sns.distplot(df_copy.label_count, bins=10, color='b');

In [ ]:
lbls_sorted = sorted(lbl_freqs.items(), key=lambda item: item[1], reverse=True)

In [ ]:
lbls_sorted[:20]

In [ ]:
ranked_lbls = L(lbls_sorted).itemgot(0)
ranked_freqs = L(lbls_sorted).itemgot(1)
ranked_lbls, ranked_freqs

In [ ]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1,1,1)
ax.plot(ranked_freqs)
ax.set_xlabel('Code IDs ranked by frequency')
ax.set_ylabel('Code Frequency')
ax.set_yscale('log');
fig.savefig('icd9_lbs_skew.pdf')

#### Check #4: Computing the min label freq for each text

In [ ]:
df_copy.head(10)

In [ ]:
df_copy['min_code_freq'] = df_copy.apply(
    lambda row: min([lbl_freqs[lbl] for lbl in row.labels.split(',')]), axis=1)

In [ ]:
df_copy['max_code_freq'] = df_copy.apply(
    lambda row: max([lbl_freqs[lbl] for lbl in row.labels.split(',')]), axis=1)

In [ ]:
df_copy['90pct_code_freq'] = df_copy.apply(
    lambda row: np.percentile([lbl_freqs[lbl] for lbl in row.labels.split(',')], 90), axis=1)

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(20,20))

for freq, axis in zip(['min_code_freq', 'max_code_freq', '90pct_code_freq'], axes):
    df_copy[freq].hist(ax=axis[0], bins=25)
    axis[0].set_xlabel(freq)
    axis[0].set_ylabel('Text Count')
    df_copy[freq].plot.density(ax=axis[1])
    axis[1].set_xlabel(freq)
    

In [ ]:
min_code_freqs = Counter(df_copy['min_code_freq'])
max_code_freqs = Counter(df_copy['max_code_freq'])
nintypct_code_freqs = Counter(df_copy['90pct_code_freq'])

In [ ]:
total_notes = L(min_code_freqs.values()).sum()
total_notes

In [ ]:
for kmin in min_code_freqs:
    min_code_freqs[kmin] = (min_code_freqs[kmin]/total_notes) * 100
    
for kmax in max_code_freqs:
    max_code_freqs[kmax] = (max_code_freqs[kmax]/total_notes) * 100
    
for k90pct in nintypct_code_freqs:
    nintypct_code_freqs[k90pct] = (nintypct_code_freqs[k90pct]/total_notes) * 100

In [ ]:
min_code_freqs = dict(sorted(min_code_freqs.items(), key=lambda item: item[0]))
max_code_freqs = dict(sorted(max_code_freqs.items(), key=lambda item: item[0]))
nintypct_code_freqs = dict(sorted(nintypct_code_freqs.items(), key=lambda item: item[0]))

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(20,20))

for axis, freq_dict, label in zip(axes, (min_code_freqs, max_code_freqs, nintypct_code_freqs), ('min', 'max', '90pct')):
    axis[0].plot(freq_dict.keys(), freq_dict.values())
    axis[0].set_xlabel(f"{label} label freq (f)")
    axis[0].set_ylabel("% of texts (P[f])");
    
    axis[1].plot(freq_dict.keys(), np.cumsum(list(freq_dict.values())))
    axis[1].set_xlabel(f"{label} code freq (f)")
    axis[1].set_ylabel("P[f<=t]");

### Steps for creating the classifier `DataLoaders` using fastai's `Transforms`:

#### 1. train/valid `splitter`:

Okay, based on the `is_valid` column of our Dataframe, let's create a splitter:

In [ ]:
def splitter(df):
    train = df.index[~df['is_valid']].tolist()
    valid = df.index[df['is_valid']].to_list()
    return train, valid

Let's check the train/valid split

In [ ]:
splits = [train, valid] = splitter(df)
L(splits[0]), L(splits[1])

#### 2. Making the `Datasets` object:

**Crucial:** We need the vocab of the language model so that we can make sure we use the same correspondence of token to index. Otherwise, the embeddings we learned in our fine-tuned language model won't make any sense to our classifier model, and the fine-tuning won't be of any use. So we need to pass the `lm_vocab` to the `Numericalize` transform:

So let's load the vocab of the language model:

In [ ]:
lm_vocab = torch.load(source/'mimic3-9k_dls_lm_vocab.pkl')
lm_vocab_r = torch.load(source/'mimic3-9k_dls_lm_vocab_r.pkl')

In [ ]:
all_equal(lm_vocab, lm_vocab_r)
L(lm_vocab)

In [ ]:
x_tfms = [Tokenizer.from_df('text'), attrgetter("text"), Numericalize(vocab=lm_vocab)]
x_tfms_r = [Tokenizer.from_df('text', ), attrgetter("text"), Numericalize(vocab=lm_vocab), reverse_text]
y_tfms = [ColReader('labels', label_delim=';'), MultiCategorize(vocab=lbls), OneHotEncode()]
tfms = [x_tfms, y_tfms]
tfms_r = [x_tfms_r, y_tfms]

In [ ]:
%%time
dsets = Datasets(df, tfms, splits=splits)

In [ ]:
%%time
dsets_r = Datasets(df, tfms_r, splits=splits)

Let's now check if our `Datasets` got created alright:

In [ ]:
len(dsets.train), len(dsets.valid)

Let's check a random data point:

In [ ]:
idx = random.randint(0, len(dsets))
x = dsets.train[idx]
assert isinstance(x, tuple) # tuple if independent and dependent variable
x_r = dsets_r.train[idx]

In [ ]:
dsets.decode(x)

In [ ]:
dsets.show(x)

In [ ]:
assert isinstance(dsets.tfms[0], Pipeline) # `Pipeline` of the `x_tfms`
assert isinstance(dsets.tfms[0][0], Tokenizer)
assert isinstance(dsets.tfms[0][1], Transform)
assert isinstance(dsets.tfms[0][2], Numericalize)

If we just want to decode the one-hot encoded dependent variable:

In [ ]:
_ind, _dep = x
_lbl = dsets.tfms[1].decode(_dep)
test_eq(_lbl, array(lbls)[_dep.nonzero().flatten().int().numpy()])

Let's extract the `MultiCategorize` transform applied by `dsets` on the dependent variable so that we can apply it ourselves:

In [ ]:
tfm_cat = dsets.tfms[1][1]
test_eq(str(tfm_cat.__class__), "<class 'fastai.data.transforms.MultiCategorize'>")

`vocab` attribute of the `MultiCategorize` transform stores the category vocabulary. If it was specified from outside (in this case it was) then the `MultiCategorize` transform will not sort the vocabulary otherwise it will.

In [ ]:
test_eq(lbls, tfm_cat.vocab)
test_ne(lbls, sorted(tfm_cat.vocab))

In [ ]:
test_eq(str(_lbl.__class__), "<class 'fastai.data.transforms.MultiCategory'>")
test_eq(tfm_cat(_lbl), TensorMultiCategory([lbls.index(o) for o in _lbl]))
test_eq(tfm_cat.decode(tfm_cat(_lbl)), _lbl)

Let's check the reverse:

In [ ]:
# dsets_r.decode(x_r)

In [ ]:
# dsets_r.show(x_r)

Looks pretty good!

#### 3. Making the `DataLoaders` object:

We need to pick the sequence length and the batch size (you might have to adjust this depending on you GPU size)

In [ ]:
bs, sl = 16, 72

We will use the `dl_type` argument of the `DataLoaders`. The purpose is to tell `DataLoaders` to use `SortedDL` class of the `DataLoader`, and not the usual one. `SortedDL` constructs batches by putting samples of roughly the same lengths into batches.

In [ ]:
dl_type = partial(SortedDL, shuffle=True)

**Crucial:** 
- We will use **`pad_input_chunk`** because our encoder `AWD_LSTM` will be wrapped inside `SentenceEncoder`. 
- A `SenetenceEncoder` expects that all the documents are padded, 
- with most of the padding at the beginning of the document, with each sequence beginning at a round multiple of bptt
- and the rest of the padding at the end.

In [ ]:
dls_clas = dsets.dataloaders(bs=bs, seq_len=sl, 
                             dl_type=dl_type,
                             before_batch=pad_input_chunk)

For the reverse:

In [ ]:
dls_clas_r = dsets_r.dataloaders(bs=bs, seq_len=sl,
                                 dl_type=dl_type,
                                 before_batch=pad_input_chunk)

Creating the `DataLoaders` object takes considerable amount of time, so do save it when working on your dataset. In this though, (as always) `untar_xxx` downloaded it for you:

In [ ]:
!tree -shLD 1 {source} -P *clas*
# or using glob
# L(source.glob("**/*clas*"))

Aside: Some handy linux find tricks:
1. https://stackoverflow.com/questions/18312935/find-file-in-linux-then-report-the-size-of-file-searched
2. https://stackoverflow.com/questions/4210042/how-to-exclude-a-directory-in-find-command

In [ ]:
# !find -path ./models -prune -o -type f -name "*caml*" -exec du -sh {} \;
# !find -not -path "./data/*" -type f -name "*caml*" -exec du -sh {} \;
# !find {path_data} -type f -name "*caml*" | xargs du -sh

If you want to load the dls for the full dataset:

In [ ]:
%%time
dls_clas = torch.load(source/'mimic3-9k_dls_clas.pkl')
dls_clas_r = torch.load(source/'mimic3-9k_dls_clas_r.pkl')

Let's take a look at the data:

In [ ]:
# dls_clas.show_batch(max_n=3)

In [ ]:
# dls_clas_r.show_batch(max_n=3)

#### 4. (Optional) Making the `DataLoaders` using fastai's `DataBlock` API:

It's worth mentioning here that all the steps we performed to create the `DataLoaders` can be packaged together using fastai's `DataBlock` API. 

In [ ]:
dblock = DataBlock(
        blocks = (TextBlock.from_df('text', seq_len=sl, vocab=lm_vocab), MultiCategoryBlock(vocab=lbls)),
        get_x = ColReader('text'),
        get_y = ColReader('labels', label_delim=';'),
        splitter = splitter,
        dl_type = dl_type,
)

In [ ]:
dls_clas = dblock.dataloaders(df, bs=bs, before_batch=pad_input_chunk)

In [ ]:
dls_clas.show_batch(max_n=5)

## `Learner` for Multi-Label Classifier Fine-Tuning 

In [ ]:
# set_seed(897997989, reproducible=True)
# set_seed(67, reproducible=True)
set_seed(1, reproducible=True)

This is where we have `dls_clas`(for the full dataset) we made in the previous section:

In [ ]:
!tree -shDL 1 {source} -P "*clas*"
# or using glob
# L(source.glob("**/*clas*"))

And this is where we have the finetuned language model:

In [ ]:
!tree -shDL 1 {source} -P '*fine*'

And this is where we have the bootstrapped brain and the label biases:

In [ ]:
!tree -shDL 1 {source_l2r} -P "*tok_lbl_info*|*p_L*"

Next we'll create a tmp directory to store results. In order for our learner to have access to the finetuned language model we need to symlink to it.

In [ ]:
#| eval: false
tmp = Path.cwd()/'tmp/models'
tmp.mkdir(exist_ok=True, parents=True)
tmp = tmp.parent
# (tmp/'models'/'mimic3-9k_lm_decoder.pth').symlink_to(source/'mimic3-9k_lm_decoder.pth') # run this just once
# (tmp/'models'/'mimic3-9k_lm_decoder_r.pth').symlink_to(source/'mimic3-9k_lm_decoder_r.pth') # run this just once
# (tmp/'models'/'mimic3-9k_lm_finetuned.pth').symlink_to(source/'mimic3-9k_lm_finetuned.pth') # run this just once
# (tmp/'models'/'mimic3-9k_lm_finetuned_r.pth').symlink_to(source/'mimic3-9k_lm_finetuned_r.pth') # run this just once
# (tmp/'models'/'mimic3-9k_tok_lbl_info.pkl').symlink_to(join_path_file('mimic3-9k_tok_lbl_info', source_l2r, ext='.pkl')) #run this just once
# (tmp/'models'/'p_L.pkl').symlink_to(join_path_file('p_L', source_l2r, ext='.pkl')) #run this just once
# (tmp/'models'/'lin_lambdarank_full.pth').symlink_to(join_path_file('lin_lambdarank_full', source_l2r, ext='.pth')) #run this just once
# list_files(tmp)
!tree -shD {tmp}

Let's now get the dataloaders for the classifier. We'll also save classifier with `fname`.

In [ ]:
fname="mimic3-9k_clas_tiny_plant"
dls_clas = torch.load(tmp/'mimic3-9k_tiny_dls_clas_16.pkl')

Let's create the saving callback upfront:

In [ ]:
fname_r = fname+'_r'
cbs=SaveModelCallback(monitor='valid_precision_at_k', fname=fname, with_opt=True, reset_on_fit=True)
cbs_r=SaveModelCallback(monitor='valid_precision_at_k', fname=fname_r, with_opt=True, reset_on_fit=True)

We will make the `TextLearner` (Here you can use `pretrained=False` to save time beacuse we are anyway going to `load_encoder` later which will replace the encoder wgts with the ones we that we have in the fine-tuned LM):

Todo(Deb): Implement TTA
- make `max_len=None` during validation
- magnify important tokens

In [ ]:
learn = xmltext_classifier_learner(dls_clas, AWD_LSTM, drop_mult=0.1, max_len=72*40, 
                                   metrics=partial(precision_at_k, k=15), path=tmp, cbs=cbs,
                                   pretrained=False,
                                   splitter=None,
                                   running_decoder=True
                                  ).to_fp16()

Note: Don't forget to check k in inattention

In [ ]:
#| hide
# learn.model[0].__class__??
# learn.model[1].__class__??
# learn.model[1].pay_attn.__class__??

A few customizations into fastai's callbacks:

To tracks metrics on training bactches during an epoch:

In [ ]:
# tell `Recorder` to track `train_metrics`
assert learn.cbs[1].__class__ is Recorder
setattr(learn.cbs[1], 'train_metrics', True)

In [ ]:
# import copy
# mets = copy.deepcopy(learn.recorder._valid_mets)
# # mets = L(AvgSmoothLoss(), AvgMetric(precision_at_k))
# rv = RunvalCallback(mets)
# learn.add_cbs(rv)
# learn.cbs

In [ ]:
@patch
def after_batch(self: ProgressCallback):
        self.pbar.update(self.iter+1)
        mets = ('_valid_mets', '_train_mets')[self.training]
        self.pbar.comment = ' '.join([f'{met.name} = {met.value.item():.4f}' for met in getattr(self.recorder, mets)])

The following line essentially captures the magic of [ULMFit's](https://arxiv.org/pdf/1801.06146.pdf) transfer learning:

In [ ]:
learn = learn.load_encoder('mimic3-9k_lm_finetuned')

In [ ]:
# learn = learn.load_brain('mimic3-9k_tok_lbl_info', 'p_L')
# learn = learn.load_diffntble_brain('mimic3-9k_tok_lbl_info', 'lin_lambdarank_full', 'mimic3-9k_lm_decoder')
# learn = learn.load_both('mimic3-9k_tok_lbl_info', 'p_L', 'lin_lambdarank_full', 'lm_now_decoder')

In [ ]:
learn = learn.load_both('mimic3-9k_tok_lbl_info', 'p_L', 'lin_lambdarank_full', 'mimic3-9k_lm_decoder')
setattr(learn, 'splitter', awd_lstm_xclas_split)
learn.create_opt()

#### Check Splitter

Before:

In [ ]:
learn.model[0].module

In [ ]:
learn.splitter??

In [ ]:
len(params(learn.model))

In [ ]:
# print('\n'.join(L(learn.model.named_parameters()).itemgot(0)))
for n,p in learn.model.named_parameters(): print(n, ":", p.shape)

In [ ]:
ic(len(learn.opt.param_lists));

In [ ]:
groups = nn.Sequential(learn.model[0].module.encoder, learn.model[0].module.encoder_dp)
ic(len(params(groups)));
test_eq(params(groups), learn.opt.param_lists[0])
ic(len(learn.opt.param_lists[0]));
for n,p in groups.named_parameters(): print(n, "--", p)

In [ ]:
groups =  L(nn.Sequential(rnn, dp) for rnn, dp in zip(learn.model[0].module.rnns, learn.model[0].module.hidden_dps))
test_eq(learn.opt.param_lists[1:4], groups.map(params))
ic(len(groups));
ic(len(params(groups[2])));
for n,p in groups[2].named_parameters():
    print(n, "--", p)

In [ ]:
test_eq(learn.opt.param_lists[4], params(learn.model[1]))
ic(len(params(learn.model[1])));
ic(len(learn.opt.param_lists[4]));
ic(learn.model[1]);
for n,p in learn.model[1].named_parameters():
    print(n, "--", p)

In [ ]:
learn.model[1]

After

In [ ]:
learn = learn.load_both('mimic3-9k_tok_lbl_info', 'p_L', 'lin_lambdarank_full', 'mimic3-9k_lm_decoder')

In [ ]:
awd_lstm_clas_split??

In [ ]:
awd_lstm_xclas_split??

In [ ]:
setattr(learn, 'splitter', awd_lstm_xclas_split)
learn.create_opt()

In [ ]:
len(params(learn.model))

In [ ]:
len(learn.opt.param_lists)

The encoder remains the same:

In [ ]:
learn.opt.param_lists.map(len)

In [ ]:
groups = nn.Sequential(learn.model[0].module.encoder, learn.model[0].module.encoder_dp)
ic(len(params(groups)));
test_eq(params(groups), learn.opt.param_lists[0])
ic(len(learn.opt.param_lists[0]));
for n,p in groups.named_parameters(): print(n, "--", p)

In [ ]:
groups =  L(nn.Sequential(rnn, dp) for rnn, dp in zip(learn.model[0].module.rnns, learn.model[0].module.hidden_dps))
test_eq(learn.opt.param_lists[1:4], groups.map(params))
ic(len(groups));
ic(len(params(groups[2])));
for n,p in groups[2].named_parameters():
    print(n, "--", p)

The decoder:

In [ ]:
learn.model[1]

The real decoder layers are `learn.opt.param_lists[-1]`

In [ ]:
len(learn.opt.param_lists[-1])

In [ ]:
test_eq(learn.opt.param_lists[-1][0], learn.model[1].label_bias) 
assert learn.opt.param_lists[-1][0] is learn.model[1].label_bias

test_eq(learn.opt.param_lists[-1][1], learn.model[1].pay_attn.plant)
assert learn.opt.param_lists[-1][1] is learn.model[1].pay_attn.plant


test_eq(learn.opt.param_lists[-1][2], learn.model[1].pay_attn.lbs.weight)
assert learn.opt.param_lists[-1][2] is  learn.model[1].pay_attn.lbs.weight
    
test_eq(learn.opt.param_lists[-1][3], learn.model[1].boost_attn.lin.weight)
assert learn.opt.param_lists[-1][3] is  learn.model[1].boost_attn.lin.weight
    
test_eq(learn.opt.param_lists[-1][4], learn.model[1].boost_attn.lin.bias)
assert learn.opt.param_lists[-1][4] is  learn.model[1].boost_attn.lin.bias

In [ ]:
test_eq(learn.opt.param_lists[-1][0], learn.model[1].label_bias)
learn.opt.param_lists[-1][0] is learn.model[1].label_bias

test_eq(learn.opt.param_lists[-1][1], learn.model[1].pay_attn.wgts)
learn.opt.param_lists[-1][1] is learn.model[1].pay_attn.wgts

test_eqs(learn.opt.param_lists[-1][2], learn.model[1].pay_attn.lbs.weight, learn.model[1].pay_attn.lin_attn.func.based_on.weight)
learn.opt.param_lists[-1][2] is learn.model[1].pay_attn.lbs.weight and learn.model[1].pay_attn.lbs.weight is learn.model[1].pay_attn.lin_attn.func.based_on.weight

test_eq(learn.opt.param_lists[-1][3], learn.model[1].pay_attn.compress.weight)
learn.opt.param_lists[-1][3] is learn.model[1].pay_attn.compress.weight

test_eq(learn.opt.param_lists[-1][4], learn.model[1].pay_attn.compress.bias)
learn.opt.param_lists[-1][4] is learn.model[1].pay_attn.compress.bias

test_eq(learn.opt.param_lists[-1][5], learn.model[1].boost_attn.lin.weight)
learn.opt.param_lists[-1][5] is learn.model[1].boost_attn.lin.weight

test_eq(learn.opt.param_lists[-1][6], learn.model[1].boost_attn.lin.bias)
learn.opt.param_lists[-1][6] is learn.model[1].boost_attn.lin.bias


The planted l2r is `learn.opt.param_lists[-2]`:

In [ ]:
test_eqs(learn.opt.param_lists[-2][0], learn.model[1].pay_attn.l2r.token_factors.weight, learn.model[1].pay_attn.attn.func.based_on.token_factors.weight)
learn.opt.param_lists[-2][0] is learn.model[1].pay_attn.l2r.token_factors.weight and learn.model[1].pay_attn.l2r.token_factors.weight is learn.model[1].pay_attn.attn.func.based_on.token_factors.weight

test_eqs(learn.opt.param_lists[-2][1], learn.model[1].pay_attn.l2r.token_bias.weight, learn.model[1].pay_attn.attn.func.based_on.token_bias.weight)
learn.opt.param_lists[-2][1] is learn.model[1].pay_attn.l2r.token_bias.weight and learn.model[1].pay_attn.l2r.token_bias.weight is learn.model[1].pay_attn.attn.func.based_on.token_bias.weight

test_eqs(learn.opt.param_lists[-2][2], learn.model[1].pay_attn.l2r.label_factors.weight, learn.model[1].pay_attn.attn.func.based_on.label_factors.weight)
learn.opt.param_lists[-2][2] is learn.model[1].pay_attn.l2r.label_factors.weight and learn.model[1].pay_attn.l2r.label_factors.weight is learn.model[1].pay_attn.attn.func.based_on.label_factors.weight

test_eqs(learn.opt.param_lists[-2][3], learn.model[1].pay_attn.l2r.label_bias.weight, learn.model[1].pay_attn.attn.func.based_on.label_bias.weight)
learn.opt.param_lists[-2][3] is learn.model[1].pay_attn.l2r.label_bias.weight and learn.model[1].pay_attn.l2r.label_bias.weight is learn.model[1].pay_attn.attn.func.based_on.label_bias.weight

The lm_decoder is `learn.opt.param_lists[-3]`

In [ ]:
test_eq(learn.opt.param_lists[-3][0], learn.model[1].pay_attn.lm_decoder.decoder.weight)
learn.opt.param_lists[-3][0] is learn.model[1].pay_attn.lm_decoder.decoder.weight

test_eq(learn.opt.param_lists[-3][1], learn.model[1].pay_attn.lm_decoder.decoder.bias)
learn.opt.param_lists[-3][1] is learn.model[1].pay_attn.lm_decoder.decoder.bias

In [ ]:
class _FakeLearner: 
    def to_detach(self,b,cpu=True,gather=True):
        return to_detach(b,cpu,gather)
_fake_l = _FakeLearner()

def cpupy(t): return t.cpu().numpy() if isinstance(t, Tensor) else t
learn.model = learn.model.to('cuda:0')

TODO:
- Also print avg text lengths

In [ ]:
# import copy
# mets = L(AvgLoss(), AvgMetric(partial(precision_at_k, k=15)))
mets = L(F1ScoreMulti(thresh=0.14, average='macro'), F1ScoreMulti(thresh=0.14, average='micro'))  #(learn.recorder._valid_mets)
learn.model.eval()
mets.map(Self.reset())
pbar = progress_bar(learn.dls.valid)
log_file = join_path_file('log', learn.path/learn.model_dir, ext='.csv')
with open(log_file, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    header = mets.attrgot('name') + L('bs', 'n_lbs', 'mean_nlbs')
    writer.writerow(header)
    for i, (xb, yb) in enumerate(pbar):
        _fake_l.yb = (yb,)
        _fake_l.y = yb
        _fake_l.pred, *_ = learn.model(xb) 
        _fake_l.loss = Tensor(learn.loss_func(_fake_l.pred, yb))
        for met in mets: met.accumulate(_fake_l)
        pbar.comment = ' '.join(mets.attrgot('value').map(str))
        yb_nlbs =  Tensor(yb.count_nonzero(dim=1)).float().cpu().numpy()
        writer.writerow(mets.attrgot('value').map(cpupy) + L(find_bs(yb), yb_nlbs, yb_nlbs.mean()))

In [ ]:
pd.set_option('display.max_rows', 100)
df = pd.read_csv(log_file)
df

In [ ]:
# learn.model[1].label_bias.data.min(),  learn.model[1].label_bias.data.max()
# nn.init.kaiming_normal_(learn.model[1].label_bias.data.unsqueeze(-1))
# # init_default??
# with torch.no_grad(): 
    # learn.model[1].label_bias.data = learn.lbsbias
# learn.model[1].label_bias.data.min(),  learn.model[1].label_bias.data.max()
# learn.model[1].pay_attn.attn.func.f#, learn_r.model[1].pay_attn.attn.func.f
# set_seed(1, reproducible=True)

## Dev (Ignore)

In [ ]:
df_des = pd.read_csv(source/'code_descriptions.csv')

In [ ]:
learn = learn.load_brain('mimic3-9k_tok_lbl_info')
learn.brain.shape

In [ ]:
L(learn.dls.vocab[0]), L(learn.dls.vocab[1])

In [ ]:
rnd_code = '733.00' #random.choice(learn.dls.vocab[1]) # '96.04'
des = load_pickle(join_path_file('code_desc', source, ext='.pkl'))
k=20
idx, *_ = dls_clas.vocab[1].map_objs([rnd_code])
print(f"For the icd code {rnd_code} which means {des[rnd_code]}, the top {k} tokens in the brain are:")
print('\n'.join(L(array(learn.dls.vocab[0])[learn.brain[:, idx].topk(k=k).indices.cpu()], use_list=True)))

Let's pull out a text from the batch and take a look at the text and its codes:

In [ ]:
xb, yb = dls_clas.one_batch()
i = 3
text = ' '.join([dls_clas.vocab[0][o] for o in xb[3] if o !=1])
codes = dls_clas.tfms[1].decode(yb[3])
print(f"The text is {len(text)} words long and has {len(codes)} codes")

In [ ]:
df_codes = pd.DataFrame(columns=['code', 'description', 'freq', 'top_toks'])
df_codes['code'] = codes
df_codes['description'] = mapt(des.get, codes)
df_codes['freq'] = mapt(lbl_freqs.get, codes)
idxs = learn.dls.vocab[1].map_objs(codes)
top_toks = array(learn.dls.vocab[0])[learn.brain[:, idxs].topk(k=k, dim=0).indices.cpu()].T
top_vals = learn.brain[:, idxs].topk(k=k, dim=0).values.cpu().T
df_codes['top_toks'] = L(top_toks, use_list=True).map(list)
df_codes

In [ ]:
print('\n'.join(L(zip(top_toks[16], top_vals[16])).map(str)))

In [ ]:
tok_list = top_toks[16]
L(tok for tok in tok_list if tok in text.split())

In [ ]:
pattern = r'\b({0})\b'.format('|'.join(list(tok_list)))
pattern = re.compile(pattern)
for i,match in enumerate(pattern.finditer(text)):
    print(i, match.group())
    print("-"*len(match.group()))
    print(text[match.start()-100: match.end()+100], end='\n\n')

In [ ]:
tst_model = get_xmltext_classifier2(AWD_LSTM, 60000, 1271, seq_len=72, config=awd_lstm_clas_config, 
                               drop_mult=0.1, max_len=72*40).to(default_device())

xb, yb = dls_clas.one_batch()

L(xb, yb).map(lambda o: (o.shape, o.device))

sent_enc = learn.model[0].to(default_device())
attn_clas = learn.model[1].to(default_device())

preds_by_sent_enc, mask = sent_enc(xb)
preds_by_sent_enc.shape, mask.shape

o = tst_model(xb)

o[0].shape

tst_model[1].hl.shape

## Fine-Tuning the Classifier 

In [ ]:
learn.fit(2, lr=3e-2)

In [ ]:
learn.fit(9, lr=3e-2)

In [ ]:
learn.freeze_to(-2)

In [ ]:
learn.fit(10, lr=1e-2)

In [ ]:
learn.fit(10, lr=1e-2)

In [ ]:
learn.freeze_to(-3)

In [ ]:
learn.fit(2, lr=1e-2)

In [ ]:
learn.freeze_to(-3)

In [ ]:
learn.fit(5, lr=1e-2)

In [ ]:
learn.unfreeze()

In [ ]:
learn.fit(12, lr=1e-6, wd=0.3)

In [ ]:
learn.fit(12, lr=1e-6, wd=0.3)

In [ ]:
learn.fit(12, lr=1e-6, wd=0.3)

In [ ]:
learn.fit(17, lr=1e-6, wd=0.3)

In [ ]:
learn.fit(12, lr=1e-6, wd=0.3)

start here:

In [ ]:
print(learn.save_model.fname)
learn.save_model.reset_on_fit=False
assert not learn.save_model.reset_on_fit
learn = learn.load(learn.save_model.fname)
validate(learn)

In [ ]:
learn.save_model.best = 0.605931198

Now unfreeze and train more if you want:

In [ ]:
learn.unfreeze()

In [ ]:
learn.fit(12, lr=1e-6, wd=0.3)

---

The last step (yes, the madness will end soon) is to train with:
- *discriminative learning rates*: define here
- *gradual unfreezing*: define here

as defined here in [ULMFit](https://arxiv.org/pdf/1801.06146.pdf)

If we are using the `LabelForcing` callback, then the loss will spike after certain number of epochs. So we need to remove this callback before doing `lr_find`. This is because `lr_find` aborts if things go south with regard to loss. 

In [ ]:
lbl_fcr = None
with suppress(ValueError): lbl_fcr = learn.cbs[learn.cbs.map(type).index(LabelForcing)]
with learn.removed_cbs(lbl_fcr):
    lr_min, lr_steep, lr_valley, lr_slide = learn.lr_find(suggest_funcs=(minimum, steep, valley, slide), num_it=5000)
    print(f"{lr_min=}, {lr_steep=}, {lr_valley=}, {lr_slide=}")

In [ ]:
# learn.opt.hypers.map(lambda d: d['lr'])

In [ ]:
[0.14*i/4 for i in range(4, 0, -1)], [0.14*i for i in range(4, 0, -1)], 

In [ ]:
learn.cbs[-4].reset_on_fit = False
assert not learn.cbs[-4].reset_on_fit
# for i in range(4, 0, -1):
#     learn.fit_one_cycle(1, lr_max=0.14*i, moms=(0.8,0.7,0.8), wd=0.1)
#     learn.fit_one_cycle(5, lr_max=0.14*i, moms=(0.8,0.7,0.8), wd=0.1)
learn.fit_sgdr(4, 1, lr_max=0.2, wd=0.1)

In [ ]:
validate(learn)

In [ ]:
learn.opt.hypers.map(lambda d: d['lr'])
# L(apply(d.get, ['wd', 'lr']))

This was the result after just three epochs. Now let's unfreeze the last two layers and do discriminative training:  

Now we will pass -2 to `freeze_to` to freeze all except the last two parameter groups:

In [ ]:
learn.freeze_to(-2)
# learn.freeze_to(-4)

In [ ]:
# learn.fit_one_cycle(1, lr_max=1e-3*i, moms=(0.8,0.7,0.8), wd=0.2)
# learn.fit_one_cycle(4, lr_max=1e-3*i, moms=(0.8,0.7,0.8), wd=0.2)
learn.fit_sgdr(3, 1, lr_max=0.02, wd=0.1)

In [ ]:
learn.opt.hypers.map(lambda d: d['lr'])
# L(apply(d.get, ['wd', 'lr']))

In [ ]:
validate(learn)

Now we will unfreeze a bit more, recompute learning rate and continue training:

In [ ]:
learn.freeze_to(-3)
# learn.freeze_to(-5)

In [ ]:
learn.fit_one_cycle(1, lr_max=1e-3, moms=(0.8,0.7,0.8), wd=0.2)
learn.fit_one_cycle(4, lr_max=1e-3, moms=(0.8,0.7,0.8), wd=0.2)

In [ ]:
learn.opt.hypers.map(lambda d: d['lr'])
# L(apply(d.get, ['wd', 'lr']))

In [ ]:
validate(learn)

Finally, we will unfreeze the whole model and perform training: (Caution: Check if you got enough memory left!)

In [ ]:
cudamem()

In [ ]:
learn.unfreeze()

At this point, it starts overfitting too soon so either progressively increase `bs` or increase `wd` or decrease `lr_max`. Another way would be to drop out label embeddings, but for this one needs to carefully adjust the dropout prob. Same concept regarding dropping out the seqs nh.

In [ ]:
lr_min, lr_steep, lr_valley, lr_slide = learn.lr_find(suggest_funcs=(minimum,steep,valley,slide), num_it=100)

In [ ]:
lr_min, lr_steep, lr_valley, lr_slide

In [ ]:
# learn.fit_one_cycle(1, lr_max=1e-3, moms=(0.8,0.7,0.8), wd=0.2)
for i in range(3, 0, -1):
    learn.fit_one_cycle(1, lr_max=1e-3*i, moms=(0.8,0.7,0.8), wd=0.1)
    learn.fit_one_cycle(5, lr_max=1e-3*i, moms=(0.8,0.7,0.8), wd=0.1)

In [ ]:
learn.opt.hypers.map(lambda d: d['lr'])
# L(apply(d.get, ['wd', 'lr']))

In [ ]:
validate(learn)

## Fine-Tune the Fwd (Dev|Tiny)

In [ ]:
learn = learn.load(learn.save_model.fname)
validate(learn)

In [ ]:
learn.save_model.reset_on_fit = False
learn.save_model.fname

**Without Gradual Unfreezing:**

In [ ]:
learn.freeze_to(-3)

In [ ]:
learn.fit(40, lr=1e-2)

**With Gradual Unfreezing:**

In [ ]:
learn.fit(2, lr=7e-2)

In [ ]:
learn.fit(3, lr=7e-2)

In [ ]:
learn.freeze_to(-2)

In [ ]:
learn.fit(10, lr=3e-2)

In [ ]:
learn.freeze_to(-3)

In [ ]:
learn.fit(10, lr=1e-3)

In [ ]:
learn.fit(10, lr=1e-3)

In [ ]:
# all_equal(*learn.opt.param_lists[:2])
# torch.equal(learn.opt.param_lists[6][0].data, learn.model[-1].pay_attn.lbs.weight.data)

In [ ]:
learn.unfreeze()

In [ ]:
learn.fit(25, lr=1e-6, wd=0.3)

## Fine-Tune the Fwd (Dev|Full)

In [ ]:
learn.opt.hypers

In [ ]:
learn.opt.set_hypers(mom=0.99)

In [ ]:
learn.opt.hypers

In [ ]:
learn.freeze_to(-2)

In [ ]:
learn.fit(10, lr=[1e-6, 1e-6, 1e-6, 1e-6, 1e-6, 7e-4, 3e-2], wd=[0.01, 0.01, 0.01, 0.01, 0.01, 0.1, 0.01])

## Checking how to avoid overfitting:

In [ ]:
learn.save_model.reset_on_fit=False

In [ ]:
learn.fit(10, lr=1e-2)

In [ ]:
learn.freeze_to(-2)

In [ ]:
learn.fit(5, lr=1e-2)

In [ ]:
learn.freeze_to(-3)

In [ ]:
learn.fit(5, lr=1e-2)

In [ ]:
learn.opt.hypers

In [ ]:
learn = xmltext_classifier_learner(dls_clas, AWD_LSTM, max_len=72*40, 
                                   metrics=partial(precision_at_k, k=15), path=tmp, cbs=cbs, 
                                   drop_mult=0.1,
                                   pretrained=False,
                                   splitter=None,
                                   running_decoder=True).to_fp16()

In [ ]:
# tell `Recorder` to track `train_metrics`
assert learn.cbs[1].__class__ is Recorder
setattr(learn.cbs[1], 'train_metrics', True)

In [ ]:
@patch
def after_batch(self: ProgressCallback):
        self.pbar.update(self.iter+1)
        mets = ('_valid_mets', '_train_mets')[self.training]
        self.pbar.comment = ' '.join([f'{met.name} = {met.value.item():.4f}' for met in getattr(self.recorder, mets)])

In [ ]:
learn = learn.load(learn.save_model.fname)
validate(learn)

In [ ]:
learn.save_model.best=0.34333

In [ ]:
learn.unfreeze()

In [ ]:
# %%debug
learn.fit(10, lr=1e-6, wd=0.3)

In [ ]:
# learn.opt.hypers, learn.wd, learn.lr

## Fine-Tuning the Bwd Classifier 

In [ ]:
# learn_r.opt.hypers#.map(lambda d: d['lr'])
# L(apply(d.get, ['wd', 'lr']))

In [ ]:
learn_r.fit(2, lr=3e-2)

In [ ]:
learn_r.fit(9, lr=3e-2)

In [ ]:
learn_r.freeze_to(-2)

In [ ]:
learn_r.fit(14, lr=1e-2)

In [ ]:
learn_r.fit(5, lr=1e-2)

In [ ]:
learn_r.freeze_to(-3)

In [ ]:
learn_r.fit(5, lr=1e-2)

In [ ]:
learn_r.unfreeze()

In [ ]:
learn_r.fit(8, lr=1e-6, wd=0.3)

In [ ]:
learn_r.fit(2, lr=1e-6, wd=0.3)

In [ ]:
learn_r.fit(10, lr=1e-6, wd=0.3)

In [ ]:
learn_r.fit(12, lr=1e-6, wd=0.3)

In [ ]:
learn_r.fit(12, lr=1e-6, wd=0.3)

In [ ]:
learn_r.fit(12, lr=1e-6, wd=0.4)

In [ ]:
learn_r.fit(10, lr=1e-6, wd=0.4)

Start here:

In [ ]:
print(learn_r.save_model.fname)
learn_r.save_model.reset_on_fit = False
assert learn_r.save_model.reset_on_fit is False
learn_r = learn_r.load(learn_r.save_model.fname)
validate(learn_r)

In [ ]:
learn_r.save_model.best = 0.60179913

Now you can unfreeze and train more if you want:

In [ ]:
learn_r.unfreeze()

Ensemble:

In [ ]:
learn = learn.load(learn.save_model.fname)
learn_r = learn_r.load(learn_r.save_model.fname)
preds, targs = learn.get_preds()
preds_r, targs = learn_r.get_preds()

In [ ]:
precision_at_k(preds.add(preds_r), targs, k=15)

---

In [ ]:
set_seed(1, reproducible=True)

In [ ]:
lr_min, lr_steep, lr_valley, lr_slide = learn_r.lr_find(suggest_funcs=(minimum, steep, valley, slide))
lr_min, lr_steep, lr_valley, lr_slide

In [ ]:
learn_r.fit_one_cycle(1, lr_max=0.14, moms=(0.8,0.7,0.8), wd=0.1)
learn_r.fit_one_cycle(5, lr_max=0.14, moms=(0.8,0.7,0.8), wd=0.1)

In [ ]:
validate(learn_r)

This was the result after just three epochs. Now let's unfreeze the last two layers and do discriminative training:  

Now we will pass -2 to `freeze_to` to freeze all except the last two parameter groups:

In [ ]:
learn_r.freeze_to(-2)

In [ ]:
learn_r.fit_one_cycle(1, lr_max=1e-3, moms=(0.8,0.7,0.8), wd=0.1)
learn_r.fit_one_cycle(4, lr_max=1e-3, moms=(0.8,0.7,0.8), wd=0.1)

In [ ]:
validate(learn_r)

Now we will unfreeze a bit more, recompute learning rate and continue training:

In [ ]:
learn_r.freeze_to(-3)

In [ ]:
learn_r.fit_one_cycle(5, lr_max=1e-3, moms=(0.8,0.7,0.8), wd=0.2)

In [ ]:
validate(learn_r)

Finally, we will unfreeze the whole model and perform training: (Caution: Check if you got enough memory left!)

In [ ]:
cudamem()

In [ ]:
learn_r.unfreeze()

At this point, it starts overfitting too soon so either progressively increase `bs` or increase `wd` or decrease `lr_max`. Another way would be to drop out label embeddings, but for this one needs to carefully adjust the dropout prob. Same concept regarding dropping out the seqs nh.

In [ ]:
lr_min, lr_steep, lr_valley, lr_slide = learn_r.lr_find(suggest_funcs=(minimum,steep,valley,slide), num_it=100)

In [ ]:
lr_min, lr_steep, lr_valley, lr_slide

In [ ]:
learn_r.fit_one_cycle(10, lr_max=1e-4, moms=(0.8,0.7,0.8), wd=0.1)

In [ ]:
validate(learn_r)

Ensemble of fwd+bwd:

In [ ]:
preds, targs = learn.get_preds()
preds_r, targs = learn_r.get_preds()
precision_at_k(preds+preds_r, targs, k=15)

## TTA

In [ ]:
learn = xmltext_classifier_learner(dls_clas, AWD_LSTM, drop_mult=0.1, max_len=None, 
                                   metrics=partial(precision_at_k, k=15), path=tmp, cbs=cbs,
                                   pretrained=False,
                                   splitter=None,
                                   running_decoder=True).to_fp16()
learn_r = xmltext_classifier_learner(dls_clas_r, AWD_LSTM, drop_mult=0.1, max_len=None, 
                                     metrics=partial(precision_at_k, k=15), path=tmp, cbs=cbs,
                                     pretrained=False,
                                     splitter=None,
                                     running_decoder=True).to_fp16()  

In [ ]:
learn = learn.load(learn.save_model.fname)
# learn = learn.load((learn.path/learn.model_dir)/'mimic3-9k_clas_tiny')
learn_r = learn_r.load(learn_r.save_model.fname)

In [ ]:
validate(learn)

In [ ]:
validate(learn_r)

In [ ]:
L(learn, learn_r).attrgot('model').itemgot(1)

In [ ]:
from xcube.layers import *
from xcube.layers import _linear_attention, _planted_attention
from xcube.text.learner import _get_text_vocab, _get_label_vocab

In [ ]:
@patch 
def load_brain(self:TextLearner,
    file_wgts: str, # Filename of the saved attention wgts
    file_bias: str, # Filename of the saved label bias
    device:(int,str,torch.device)=None # Device used to load, defaults to `dls` device
    ):
    """Load the pre-learnt label specific attention weights for each token from `file` located in the 
    model directory, optionally ensuring it's one `device`
    """
    brain_path = join_path_file(file_wgts, self.path/self.model_dir, ext='.pkl')
    bias_path = join_path_file(file_bias, self.path/self.model_dir, ext='.pkl')
    brain_bootstrap = torch.load(brain_path, map_location=default_device() if device is None else device)
    *brain_vocab, brain = mapt(brain_bootstrap.get, ['toks', 'lbs', 'mutual_info_jaccard'])
    brain_vocab = L(brain_vocab).map(listify)
    vocab = L(_get_text_vocab(self.dls), _get_label_vocab(self.dls)).map(listify)
    brain_bias = torch.load(bias_path, map_location=default_device() if device is None else device)
    brain_bias = brain_bias[:, :, 0].squeeze(-1)
    print("Performing brainsplant...")
    self.brain, self.lbsbias, *_ = brainsplant(vocab, brain_vocab, brain, brain_bias)
    print("Successfull!")
    # import pdb; pdb.set_trace()
    plant_attn_layer = Lambda(Planted_Attention(self.brain))
    setattr(self.model[1].pay_attn, 'plant_attn', plant_attn_layer)
    assert self.model[1].pay_attn.plant_attn.func.f is _planted_attention
    return self

In [ ]:
# learn = learn.load_brain('mimic3-9k_tok_lbl_info', 'p_L')
learn_r = learn_r.load_brain('mimic3-9k_tok_lbl_info', 'p_L')

In [ ]:
L(learn, learn_r).attrgot('model').itemgot(1)

In [ ]:
@patch
def forward(self:XMLAttention, inp, sentc, mask):
    # sent is the ouput of SentenceEncoder i.e., (bs, max_len tokens, nh)
    test_eqs(inp.shape, sentc.shape[:-1], mask.shape)
    top_tok_attn_wgts = F.softmax(self.attn(sentc), dim=1).masked_fill(mask[:,:,None], 0) # lbl specific wts for each token (bs, max_len, n_lbs)
    attn_wgts = self.plant_attn(inp).masked_fill(mask[:,:,None], 0) #shape (bs, bptt, n_lbs)
    top_lbs_attn_wgts = attn_wgts.inattention(k=1, sort_dim=-1, sp_dim=1) # applying `inattention` across the lbs dim
    lbs_cf = top_lbs_attn_wgts.sum(dim=1) #shape (bs, n_lbs)
    # pdb.set_trace()
    return lincomb(sentc, wgts=top_tok_attn_wgts.transpose(1,2)), top_tok_attn_wgts, lbs_cf # for each lbl do a linear combi of all the tokens based on attn_wgts (bs, num_lbs, nh)

In [ ]:
from xcube.text.models.core import _pad_tensor

In [ ]:
@patch
def forward(self:LabelAttentionClassifier, sentc):
    if isinstance(sentc, tuple): inp, sentc, mask = sentc # sentc is the stuff coming outta SentenceEncoder i.e., shape (bs, max_len, nh) in other words the concatenated output of the AWD_LSTM
    test_eqs(inp.shape, sentc.shape[:-1], mask.shape)
    sentc = sentc.masked_fill(mask[:, :, None], 0)
    attn, wgts, lbs_cf = self.pay_attn(inp, sentc, mask) #shape attn: (bs, n_lbs, n_hidden), wgts: (bs, max_len, n_lbs), lbs_cf : (bs, n_lbs)
    attn = self.boost_attn(attn) # shape (bs, n_lbs, n_hidden)
    bs = self.hl.size(0)
    self.hl = self.hl.to(sentc.device)
    pred = self.hl + _pad_tensor(attn.sum(dim=2), bs) + self.label_bias  # shape (bs, n_lbs)

    if self.y_range is not None: pred = sigmoid_range(pred, *self.y_range)
    return pred, attn, wgts, lbs_cf 

In [ ]:
@patch
def after_pred(self:RNNCallback): self.learn.pred,self.raw_out,self.out, _ = [o[-1] if is_listy(o) else o for o in self.pred]

In [ ]:
from xcube.layers import inattention

In [ ]:
class FetchCallback(Callback):
    order=RNNCallback.order-1
    "Save the preds, attention wgts and labels confidence outputs"
    
    def __init__(self, predslog, rank): store_attr()
    
    def before_validate(self):
        self.predslog =  open(self.predslog, 'wb')
        self.csv_rank =  open(self.rank, 'w', newline='')
        self.writer_rank = csv.writer(self.csv_rank)
        self.writer_rank.writerow(L('subset', '#true (t)', '#true^subset (ts)', '% (ts/t)', '#top_preds', '#top_preds^subset', '#true^top_preds (ttp)', '#top_preds^subset^true', 'precision (ttp/15)', '#true^topk (ttk)', 'prec (ttk/15)'))
    
    @staticmethod
    def _get_stat(y, cf, preds):
        subset = cf>0
        truth = y.sum()
        num_true_subset = torch.logical_and(y == 1, subset).sum()
        assert preds.logical_and(tensor(1)).all() # check preds does not have any zeros
        top_preds = preds.inattention(k=14)
        num_top_preds = torch.as_tensor(top_preds.nonzero().flatten().numel())
        num_top_preds_subset = torch.logical_and(top_preds, subset).sum() #
        num_true_top_preds = torch.logical_and(y==1, top_preds).sum()
        num_top_preds_subset_true = torch.logical_and(top_preds, subset).logical_and(y==1).sum()#
        topk = preds.topk(k=15).indices
        num_true_topk = y[topk].sum()
        _d = dict(subsum = subset.sum(), num_truth = truth, num_true_subset = num_true_subset, pct_true_subset = num_true_subset.div(truth).mul(100), 
                    num_top_preds = num_top_preds, num_top_preds_subset = num_top_preds_subset, true_top_preds = num_true_top_preds, 
                    num_top_preds_subset_also_true =  num_top_preds_subset_true, 
                    prec_shdbe = torch.min(num_true_top_preds.div(15.0), torch.tensor(1)), 
                    num_true_top_k=num_true_topk, prec=num_true_topk.div(15))
        # pdb.set_trace()
        return L(_d.values()).map(Self.item())
    
    def after_pred(self): 
        self.actual_pred, self.attn, self.wgts, self.lbs_cf = [o[-1] if is_listy(o) else o for o in self.pred]
        for y,cf, preds in zip(self.y, self.lbs_cf, self.actual_pred):
            # self.writer_predslog.writerow(L(y, cf, preds).map(cpupy))
            # self.csv_predslog.flush()
            pickle.dump((y,cf,preds), self.predslog)
            self.writer_rank.writerow(FetchCallback._get_stat(y, cf, preds))
            self.csv_rank.flush()

    def after_validate(self): 
        self.predslog.close(); 
        self.csv_rank.close()

In [ ]:
validate(learn)

In [ ]:
validate(learn_r)

In [ ]:
predslog = join_path_file(fname+'_predslog', learn.path/learn.model_dir, ext='.pkl')
rank = join_path_file(fname+'_rank', learn.path/learn.model_dir, ext='.csv')
fc = FetchCallback(predslog, rank)

In [ ]:
print(f"pid = {os.getpid()}")
with learn.added_cbs([fc]):
    validate(learn)

In [ ]:
with open(Path.cwd()/'tmp.pkl', 'wb') as pf:
    for _ in range(7):
        tl = [torch.randn(5) for _ in range(3)]
        print(tl)
        pickle.dump(tl, pf)

In [ ]:
# with open(Path.cwd()/'tmp.pkl', 'rb') as f:
with open(predslog, 'rb') as f:
    while True:
        try:
            obj = pickle.load(f)
            print(obj)
        except EOFError: break

In [ ]:
datetime.fromtimestamp(predslog.stat().st_mtime).time()

In [ ]:
with open(predslog, 'rb') as preds_file, open(rank, 'w', newline='') as rank_file:
    writer = csv.writer(rank_file)
    writer.writerow(L('subset', '#true (t)', '#true^subset (ts)', '% (ts/t)', '#top_preds', '#top_preds^subset', '#true^top_preds (ttp)', '#top_preds^subset^true', 'precision (ttp/15)', '#true^topk (ttk)', 'prec (ttk/15)'))
    while True:
        try:
            y, cf, preds = pickle.load(preds_file)
            writer.writerow(FetchCallback._get_stat(y, cf, preds))
            rank_file.flush()
        except EOFError: break

In [ ]:
datetime.fromtimestamp(rank.stat().st_mtime).time()

In [ ]:
df_rank = pd.read_csv(rank)
print(f"{len(df_rank)=}")
df_rank.head()

We are missing out on:

In [ ]:
df_rank[df_rank['#true^top_preds (ttp)'] != df_rank['#true^topk (ttk)']]

In [ ]:
df_rank[df_rank['#true^top_preds (ttp)'] - df_rank['#top_preds^subset^true'] >= 1]

TODO:
- Implement sortish=shuffle+split+sort+cat

In [ ]:
df_rank['precision (ttp/15)'].mean(), df_rank['prec (ttk/15)'].mean()

In [ ]:
df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Age': [25, 30, 35, 40],
    'Salary': [50000, 60000, 70000, 80000]
})
df.style.applymap(lambda _: f"color:{'blue'}", subset=['Salary', 'Name'])

In [ ]:
# df.loc[df['precision (tpc/t)'] > 1.0, 'precision (tpc/t)'] = 1.0
# df.head()
# df.loc[:, 'precision (tpc/t)'].mean(), df['prec'].mean()

In [ ]:
tbar = tqdm(learn.dls.valid, leave=False) 
for xb,yb in tbar:
    tbar.set_postfix({'xb':xb.shape, 'yb': yb.shape})
    pred, *_ = learn.model(xb)
    prec = precision_at_k(pred, yb)
    print(prec)

## Plotting the Label Embeddings

> To know if the model learnt embeddings that are meaningful

In [ ]:
lbs_emb = learn.model[-1].pay_attn.lbs.weight
X = to_np(lbs_emb)

Now let's do a PCA and t-SNE on the labels embedding. Before doing PCA though we need to standardize `X`:

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X_stand = StandardScaler().fit_transform(X)

In [ ]:
X_reduced, _vars = plot_reduction(X, tSNE=True) 
# X_random, _vars_rnd = plot_reduction(np.random.normal(size=(X.shape[0], 400)), tSNE=True) # to compare with a random embeddings

In [ ]:
# print('\n'.join(L(source.glob('**/*desc*')).map(str)))
codes = load_pickle(source/'code_desc.pkl')
df_lbl = pd.DataFrame([(lbl, codes.get(lbl, "not found"), freq) for lbl, freq in lbl_freqs.items()], columns=['label', 'description', 'frequency'])
df_lbl = df_lbl.sort_values(by='frequency', ascending=False, ignore_index=True)
df_lbl.head()

Instead of doing a PCA on all the labels, let's do it on the `top` most frequent labels:

In [ ]:
top = 100
top_lbs = [k for k, v in lbl_freqs.most_common(top)]
tfm_cat = dls_clas.tfms[1][1]
top_lbs_emb = lbs_emb[tfm_cat(top_lbs)]
topX = to_np(top_lbs_emb)

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
topX_tsne = TSNE(n_components=2, perplexity=40).fit_transform(topX)

In [ ]:
fig = plt.figure(figsize=(12,12))
ax = fig.add_subplot(1, 1, 1)
plt.scatter(topX_tsne[:, 0], topX_tsne[:, 1], marker='x', s=5)
for lbl, x, y in zip(top_lbs, topX_tsne[:, 0], topX_tsne[:, 1]):
    plt.annotate(lbl,
                xy=(x,y),
                xytext=(5,-5),
                textcoords='offset points',
                size=7)
plt.show()

Take a look at some of the closely clustered labels to see if they have the same meaning. This will tell us if the model learned meaningful label embeddings:

In [ ]:
df_lbl[df_lbl.label.isin(['36.13', '995.91', '88.72', '37.22'])]

Looks like it did!